# Phase 7 — Arrhenius / super-Arrhenius test of the tipping rate

Resamples selected WeirdChat patterns at several **sampling temperatures** via
OpenRouter, grades with the reference rubric judge, and fits the per-token
tipping hazard `h(T)` to decide **Arrhenius vs super-Arrhenius** (a finite
critical temperature `T0` where the rate would vanish).

CPU runtime is enough (all API-bound). Needs a `OPENROUTER_API_KEY` secret
(~a few $). The sweep is resumable and writes to Drive; the analysis is
seconds. Start small (defaults below); widen temperatures/samples once the
curve shape is clear.

In [ ]:
# 1) Setup
import os
if not os.path.exists('/content/WeirdChat'):
    !git clone --branch claude/repo-published-weights-u71yew https://github.com/Erikiss/WeirdChat /content/WeirdChat
else:
    !git -C /content/WeirdChat pull
%cd /content/WeirdChat/examples/03_deepspec_draft_surprise
%pip install -q -e /content/WeirdChat

from google.colab import drive; drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/weirdspec/data'
os.makedirs(DATA, exist_ok=True); os.environ['DATA'] = DATA
try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    print('tokens loaded')
except Exception:
    from getpass import getpass
    os.environ.setdefault('OPENROUTER_API_KEY', getpass('OpenRouter key: '))

In [ ]:
# 2) Temperature sweep (resumable). First step: one surface-form behavior
#    (language switching) vs one fluent control (chemtrails). ~a few $.
!python phase7_sweep.py --output $DATA/temp_sweep.jsonl \
    --behaviors language-switching-english chemtrails-assertion \
    --patterns-per-behavior 3 --prompts-per-pattern 4 --samples-per-prompt 12 \
    --temperatures 0.5 0.7 0.85 1.0 1.15 1.3

In [ ]:
# 3) Arrhenius / super-Arrhenius diagnosis (CPU, seconds)
!python phase7_arrhenius.py --sweep $DATA/temp_sweep.jsonl \
    --output $DATA/arrhenius_report.md --key behavior_id

from IPython.display import Markdown, display
display(Markdown(open(os.environ['DATA'] + '/arrhenius_report.md').read()))

In [ ]:
# 4) (Optional) Arrhenius plot: ln(hazard) vs 1/T per behavior
import json, math
import matplotlib.pyplot as plt
from phase7_arrhenius import aggregate, diagnose

rows = [json.loads(l) for l in open(f'{DATA}/temp_sweep.jsonl') if l.strip()]
groups = aggregate(rows, 'behavior_id')
plt.figure(figsize=(6,4))
for name, pts in groups.items():
    r = diagnose(pts, n_boot=200)
    if 'ln_hazard' not in r: continue
    xs = [1.0/t for t in r['temps']]
    plt.plot(xs, r['ln_hazard'], 'o-', label=f"{name} [{r['verdict'].split()[0]}]")
plt.xlabel('1 / T'); plt.ylabel('ln per-token hazard')
plt.title('Arrhenius plot (straight=Arrhenius, concave=super-Arrhenius)')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Reading the result

- **SUPER-ARRHENIUS** for language switching (concave plot, `T0>0` inside/near
  the sampled range, VFT preferred by AIC) with **ARRHENIUS** for the fluent
  control would confirm the two-axes picture thermodynamically: surface-form
  tipping has a critical temperature, fluent weirdness does not.
- **INCONCLUSIVE** usually means too few temperatures, or the rate never leaves
  the detection floor/ceiling in the sampled window — widen `--temperatures`
  (add colder points like 0.3, 0.4) and raise `--samples-per-prompt`.
- `fragility` >> 1 is the fragile/super-Arrhenius signature; ~1 is Arrhenius.